# Analiza slovenskega nepremičninskega trga (2015–2025)
### 10 največjih občin — ETN podatki GURS

**Vir:** Geodetska uprava RS — Evidenca trga nepremičnin (ETN)  
**Filter:** Samo tržni posli | površina > 5 m² | cena > 0 | 300 < €/m² < 15.000  
**Tip:** Bivanjske nepremičnine (stanovanja + hiše)

## Nalaganje in priprava podatkov
> **Zaženi to celico enkrat pred vsemi ostalimi.** Vse vizualizacije spodaj predpostavljajo, da je `df` že naložen.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

# ── Konstante ────────────────────────────────────────────────────────────────
DATA_PATH = "clean.csv"

OBCINE = [
    "LJUBLJANA", "MARIBOR", "CELJE", "KOPER", "KRANJ",
    "DOMŽALE", "NOVO MESTO", "VELENJE", "KAMNIK", "NOVA GORICA"
]

# Barvna paleta — konsistentna čez vse grafe
BARVE = {
    "LJUBLJANA":   "#E53935",
    "MARIBOR":     "#1E88E5",
    "CELJE":       "#43A047",
    "KOPER":       "#FB8C00",
    "KRANJ":       "#8E24AA",
    "DOMŽALE":     "#00ACC1",
    "NOVO MESTO":  "#F4511E",
    "VELENJE":     "#6D4C41",
    "KAMNIK":      "#546E7A",
    "NOVA GORICA": "#FFB300",
}

# ── Nalaganje ────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

# ── Klasifikacija tipa nepremičnine ─────────────────────────────────────────
def klasificiraj_tip(raba):
    r = str(raba).strip()
    if r.startswith("111") or r.startswith("1 -") or r.startswith("1-"):
        return "Hiša"
    elif (r.startswith("112") or r.startswith("2 -") or r.startswith("2-")
          or r.startswith("47") or r.startswith("3 -")):
        return "Stanovanje"
    else:
        return "Drugo"

df_raw["TIP"] = df_raw["DEJANSKA_RABA_DELA_STAVBE"].apply(klasificiraj_tip)

# ── Izluščitev leta iz datuma ────────────────────────────────────────────────
df_raw["DATUM"] = pd.to_datetime(df_raw["DATUM_UVELJAVITVE"], dayfirst=True, errors="coerce")
df_raw["LETO"] = df_raw["DATUM"].dt.year

# ── Filtri ───────────────────────────────────────────────────────────────────
df = df_raw[
    df_raw["OBCINA"].isin(OBCINE) &
    (df_raw["POVRSINA"] > 5) &
    (df_raw["POGODBENA_CENA_ODSKODNINA"] > 0) &
    (df_raw["CENA_M2"] > 300) &
    (df_raw["CENA_M2"] < 15_000) &
    df_raw["TIP"].isin(["Hiša", "Stanovanje"]) &
    df_raw["LETO"].between(2015, 2025)
].copy()

df["OBCINA"] = pd.Categorical(df["OBCINA"], categories=OBCINE, ordered=True)

print(f"Poslov po filtriranju: {len(df):,}")
print(f"Hiše: {(df['TIP']=='Hiša').sum():,}  |  Stanovanja: {(df['TIP']=='Stanovanje').sum():,}")
print(f"\nPo občinah:")
print(df.groupby("OBCINA", observed=True).size().to_string())

Poslov po filtriranju: 27,792
Hiše: 1,656  |  Stanovanja: 26,136

Po občinah:
OBCINA
LJUBLJANA      12606
MARIBOR         5977
CELJE           1852
KOPER           1159
KRANJ           1704
DOMŽALE         1053
NOVO MESTO       860
VELENJE         1206
KAMNIK           756
NOVA GORICA      619


---
## Vizualizacija 1: Gibanje cen/m² v top 10 občinah (2015–2025)

**Interaktivno:** dropdown za tip nepremičnine, legenda za vklop/izklop občin, nacionalno povprečje kot referenčna črta.  
**Opomba za 2025:** podatki za 2025 so delni (ETN ima zamik ~3–6 mesecev), zato je zadnja točka prikazana s črtkano črto.

In [2]:
# ── Vizualizacija 1: Gibanje cen/m² — interaktivni Plotly ───────────────────
# Privzeto vidne: Ljubljana, Maribor, Koper; ostale legendonly

LETA = list(range(2015, 2026))
TIPI = ["Stanovanje", "Hiša", "Skupaj"]

# Občine, ki so privzeto vidne
PRIVZETO_VIDNE = {"LJUBLJANA", "MARIBOR", "KOPER"}

def pripravi_podatke(tip_filter):
    """Vrne mediane cen/m² po občinah in letih ter nacionalno povprečje."""
    if tip_filter == "Skupaj":
        pod = df
    else:
        pod = df[df["TIP"] == tip_filter]

    # Po občinah
    obcina_leto = (
        pod.groupby(["OBCINA", "LETO"], observed=True)["CENA_M2"]
        .median()
        .reset_index()
    )

    # Nacionalno povprečje (mediana vseh 10 občin skupaj)
    nat = (
        pod.groupby("LETO")["CENA_M2"]
        .median()
        .reset_index()
        .rename(columns={"CENA_M2": "NAT"})
    )
    return obcina_leto, nat

def naredi_traces(tip_filter):
    """Izdela seznam Plotly traces za dani tip."""
    obcina_leto, nat = pripravi_podatke(tip_filter)
    traces = []

    for obcina in OBCINE:
        pod = obcina_leto[obcina_leto["OBCINA"] == obcina].sort_values("LETO")
        leta = pod["LETO"].tolist()
        cene = pod["CENA_M2"].tolist()

        # Normalna črta (2015–2024)
        idx_24 = [i for i, l in enumerate(leta) if l <= 2024]
        idx_25 = [i for i, l in enumerate(leta) if l >= 2024]

        vidnost = True if obcina in PRIVZETO_VIDNE else "legendonly"

        # 2015–2024: polna črta
        traces.append(go.Scatter(
            x=[leta[i] for i in idx_24],
            y=[cene[i] for i in idx_24],
            name=obcina.title(),
            line=dict(color=BARVE[obcina], width=2),
            mode="lines+markers",
            marker=dict(size=6),
            visible=vidnost,
            legendgroup=obcina,
            showlegend=True,
            hovertemplate=f"<b>{obcina.title()}</b><br>Leto: %{{x}}<br>Mediana: %{{y:,.0f}} €/m²<extra></extra>",
        ))

        # 2024–2025: črtkana črta (delni podatki)
        if len(idx_25) >= 2:
            traces.append(go.Scatter(
                x=[leta[i] for i in idx_25],
                y=[cene[i] for i in idx_25],
                name=obcina.title() + " (2025*)",
                line=dict(color=BARVE[obcina], width=2, dash="dot"),
                mode="lines+markers",
                marker=dict(size=6, symbol="diamond"),
                visible=vidnost,
                legendgroup=obcina,
                showlegend=False,
                hovertemplate=f"<b>{obcina.title()} (delni podatki 2025)</b><br>Leto: %{{x}}<br>Mediana: %{{y:,.0f}} €/m²<extra></extra>",
            ))

    # Nacionalno povprečje
    nat_s = nat.sort_values("LETO")
    traces.append(go.Scatter(
        x=nat_s["LETO"].tolist(),
        y=nat_s["NAT"].tolist(),
        name="🇸🇮 Nacionalno povprečje",
        line=dict(color="#212121", width=3, dash="dash"),
        mode="lines",
        visible=True,
        legendgroup="nat",
        hovertemplate="<b>Nacionalno povprečje</b><br>Leto: %{x}<br>Mediana: %{y:,.0f} €/m²<extra></extra>",
    ))

    return traces

# ── Začetni tip: Stanovanje ──────────────────────────────────────────────────
zacetni_tip = "Stanovanje"
traces_init = naredi_traces(zacetni_tip)

fig = go.Figure(data=traces_init)

# ── Dropdown gumbi ───────────────────────────────────────────────────────────
# Za vsak tip predizračunamo y vrednosti vseh trace-ov
def y_za_tip(tip_filter):
    obcina_leto, nat = pripravi_podatke(tip_filter)
    ys = []
    for obcina in OBCINE:
        pod = obcina_leto[obcina_leto["OBCINA"] == obcina].sort_values("LETO")
        cene = pod.set_index("LETO")["CENA_M2"]
        leta_24 = [l for l in LETA if l <= 2024]
        leta_25 = [2024, 2025]

        ys.append([cene.get(l, None) for l in leta_24])
        ys.append([cene.get(l, None) for l in leta_25])

    nat_s = nat.set_index("LETO")["NAT"]
    ys.append([nat_s.get(l, None) for l in LETA])
    return ys

gumbi = []
for tip in TIPI:
    ys = y_za_tip(tip)
    gumbi.append(dict(
        label=tip,
        method="update",
        args=[
            {"y": ys},
            {"title": f"Gibanje mediane cen/m² — {tip} | top 10 občin (2015–2025)"}
        ]
    ))

fig.update_layout(
    title=f"Gibanje mediane cen/m² — {zacetni_tip} | top 10 občin (2015–2025)",
    xaxis=dict(title="Leto", tickmode="linear", dtick=1, gridcolor="#eeeeee"),
    yaxis=dict(
        title="Mediana cene (€/m²)",
        tickformat=",.0f",
        ticksuffix=" €",
        gridcolor="#eeeeee",
    ),
    updatemenus=[dict(
        buttons=gumbi,
        direction="down",
        showactive=True,
        x=0.01, xanchor="left",
        y=1.13, yanchor="top",
        bgcolor="white",
        bordercolor="#cccccc",
    )],
    legend=dict(
        title="Občina<br><i>(klikni za vklop/izklop)</i>",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#cccccc",
        borderwidth=1,
    ),
    hovermode="x unified",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=12),
    width=1000,
    height=580,
    annotations=[
        dict(
            text="* 2025: delni podatki (ETN zamik ~3–6 mesecev) — črtkana črta",
            xref="paper", yref="paper",
            x=0, y=-0.1, showarrow=False,
            font=dict(size=10, color="gray"),
        ),
        dict(
            text="Vir: GURS — Evidenca trga nepremičnin (ETN)",
            xref="paper", yref="paper",
            x=1, y=-0.1, showarrow=False,
            font=dict(size=10, color="gray"),
            xanchor="right",
        ),
    ],
)

fig.write_html("viz1_cene_trend.html")
fig.show()
print("✓ Izvoženo: viz1_cene_trend.html")

✓ Izvoženo: viz1_cene_trend.html
